In previous notebook, add cell type assignments to info table, then don't need the metadata table save.

In [1]:
library(BSgenome.Hsapiens.UCSC.hg38)
library(patchwork)
library(readr)
library(stringr)
library(dplyr)
set.seed(1234)

#### Send to channel code
library(parallel)
library(plyr)
library(ggpubr)
library(car)
library(qvalue)

library(pheatmap)
library(RColorBrewer)
library(beeswarm)

Loading required package: BSgenome

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tapply,
    union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomeInfoDb

Loading required package: GenomicRanges

Loading required package: Biostrings

Loading 

In [2]:
in_dir <- '/nfs/lab/welison/multiome/chromvar/240209_ChromVar_Outputs/'

#Load in devscores, info, motif
motifdata <- read.table(file=paste0(in_dir,'motifdata.txt'), sep="\t")
info <- read.table(file=paste0(in_dir,'info_table.txt'))
TFClass_Lookup <- read_csv("/nfs/lab/welison/References/220907_WE_Chromvar_to_Gene_By_Subfam_Complete(JAPRAR2022_TFClass).csv")
TFClass_Full <- read_csv("/nfs/lab/welison/References/220907_WE_Chromvar_to_Gene_Jaspar2022.csv")
variability <- read.table(file=paste0(in_dir,'variability.txt'), sep="\t")

head(motifdata)
head(info)
dim(TFClass_Lookup)
head(TFClass_Lookup)

head(variability)
dim(variability)

Rows: 10368 Columns: 4
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (4): full_jaspar_motif, jaspar_motif, lowest_level_family, gene

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1669 Columns: 20
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (20): full_jaspar_motif, jaspar_motif, jaspar_name_1, jaspar_name_2, TFC...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


,V1,V2,V3
,<chr>,<chr>,<chr>
1,MA0030.1_FOXF2,FOXF2,Fork head/winged helix factors
2,MA0031.1_FOXD1,FOXD1,Fork head/winged helix factors
3,MA0051.1_IRF2,IRF2,Tryptophan cluster factors
4,MA0059.1_MAX::MYC,MAX::MYC,Basic helix-loop-helix factors (bHLH)
5,MA0066.1_PPARG,PPARG,Nuclear receptors with C4 zinc fingers
6,MA0069.1_PAX6,PAX6,Paired box factors


,def,cells,groups,cluster,samples,FRIP,TSS.enrichment,nCount_ATAC,nFeature_ATAC
,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<int>,<int>
MM_339_AAACGAATCCCGAAGC-1,MM_339,MM_339_AAACGAATCCCGAAGC-1,earlyT1D,Acinar_1_2_6,6247,0.5421319,4.928013,748,715
MM_339_AAACGAATCTATGAGC-1,MM_339,MM_339_AAACGAATCTATGAGC-1,earlyT1D,Acinar_1_2_6,6247,0.7366399,5.622038,830,802
MM_339_AAACTCGGTGTGCTTA-1,MM_339,MM_339_AAACTCGGTGTGCTTA-1,earlyT1D,Acinar_1_2_6,6247,0.6134342,4.720066,505,489
MM_339_AAAGATGTCGGATGTT-1,MM_339,MM_339_AAAGATGTCGGATGTT-1,earlyT1D,Acinar_1_2_6,6247,0.6927803,4.036170,746,726
MM_339_AAAGGATAGGAGTCTG-1,MM_339,MM_339_AAAGGATAGGAGTCTG-1,earlyT1D,Acinar_1_2_6,6247,0.7235805,5.164169,621,602
MM_339_AAAGGATAGTCGAGCA-1,MM_339,MM_339_AAAGGATAGTCGAGCA-1,earlyT1D,Acinar_1_2_6,6247,0.5906291,4.659066,834,797


[1] 10368     4

full_jaspar_motif,jaspar_motif,lowest_level_family,gene
<chr>,<chr>,<chr>,<chr>
MA0030.1_FOXF2,FOXF2,FOXF,FOXF2
MA0030.1_FOXF2,FOXF2,FOXF,FOXF1
MA0031.1_FOXD1,FOXD1,FOXD,FOXD1
MA0031.1_FOXD1,FOXD1,FOXD,FOXD2
MA0031.1_FOXD1,FOXD1,FOXD,FOXD3
MA0031.1_FOXD1,FOXD1,FOXD,FOXD4L1


,name,variability,bootstrap_lower_bound,bootstrap_upper_bound,p_value,p_value_adj
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MA0030.1_FOXF2,FOXF2,1.203838,1.198981,1.208643,0.000000e+00,0.000000e+00
MA0031.1_FOXD1,FOXD1,1.258115,1.252999,1.263342,0.000000e+00,0.000000e+00
MA0051.1_IRF2,IRF2,1.113662,1.108953,1.118249,0.000000e+00,0.000000e+00
MA0059.1_MAX::MYC,MAX::MYC,1.035252,1.031647,1.038709,1.326763e-95,1.932884e-95
MA0066.1_PPARG,PPARG,1.018128,1.014559,1.021752,6.234495e-27,7.676638e-27
MA0069.1_PAX6,PAX6,1.054041,1.049918,1.058356,2.990752e-220,5.306667e-220


[1] 692   6

In [3]:
# This is the big one; the deviation scores. Runs slow
dev_file <- paste0(in_dir,'devscores.txt')

devscores <- vroom::vroom(file=dev_file, skip=1, col_names=FALSE)
devscores <- tibble::column_to_rownames(devscores, var="X1")
colnames(devscores) <- str_split(readLines(file(dev_file),n=1), " ")[[1]]

dim(devscores)
head(devscores)

Rows: 692 Columns: 174599
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: " "
chr      (1): X1
dbl (174598): X2, X3, X4, X5, X6, X7, X8, X9, X10, X11, X12, X13, X14, X15, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1]    692 174598

,MM_339_AAACGAATCCCGAAGC-1,MM_339_AAACGAATCTATGAGC-1,MM_339_AAACTCGGTGTGCTTA-1,MM_339_AAAGATGTCGGATGTT-1,MM_339_AAAGGATAGGAGTCTG-1,MM_339_AAAGGATAGTCGAGCA-1,MM_339_AAAGGGCCAGTCAGCC-1,MM_339_AAAGGGCTCGATATGC-1,MM_339_AAAGGGCTCGGTTAGT-1,MM_339_AAATGAGGTGCCAAGA-1,⋯,MM_391_GTGCCAGCAGGGTAAC-1,MM_460_AAATGCCGTGCATTGT-1,MM_460_GCACCTTCATGCGCTG-1,MM_460_GTAATCGCAAAGAAGG-1,MM_460_TAGTCCCAGAAAGCAG-1,MM_460_TTCATTGTCGCAAACT-1,MM_536_GGAGGATCATCCCTCA-1,MM_544_CTAGCGGGTTTAGACC-1,MM_546_GCACGGTCAGTCAGCC-1,MM_546_TTGCAGAGTTCTGAGT-1
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MA0030.1_FOXF2,0.54520151,0.40990183,3.35607428,2.000801239,0.9658235,-0.459021816,0.44334913,-0.25558497,-1.6471011,0.8835954,⋯,-1.3485672,-1.24054617,0.6432168,0.53422671,-0.8512291,0.34641291,-1.3156336,-0.1606987,-0.4841276,-0.1220044
MA0031.1_FOXD1,-0.64639620,0.56733851,2.16471443,0.222432873,0.1040411,-1.840425687,1.07262678,-0.05391174,1.3205662,0.8736271,⋯,-2.0031346,-1.47942813,-0.2572786,-0.84672181,-1.4112522,-1.44849245,-1.0407649,-0.3555410,-2.0004751,-1.9848008
MA0051.1_IRF2,-0.04734576,0.21278816,-1.17043162,0.006864465,-0.6147434,0.779877039,-0.95418391,0.84427104,-0.1533793,-0.6127317,⋯,0.3081769,0.92292903,1.0573372,0.55484583,-0.8093365,1.63052376,0.6905937,-1.1745899,3.0360215,0.9012208
MA0059.1_MAX::MYC,-0.01129497,0.06407827,-0.60875582,-0.568995229,1.0334839,0.008193637,-1.30026440,-0.93085128,-2.0120367,-0.2730295,⋯,-1.1595347,0.42933240,2.3681214,-1.55277513,-1.3609323,-1.10154128,-1.9283999,-0.4375171,-0.9435472,-1.2343960
MA0066.1_PPARG,0.51365043,-0.50851101,-0.55188880,-0.488296947,-0.6396850,0.131552195,-0.30285472,1.31740292,1.8427194,0.4069058,⋯,1.3590695,0.03172931,0.1511277,0.35230581,-1.8023297,-0.83032002,0.4737512,-0.1385413,-0.9820997,-0.3458552
MA0069.1_PAX6,-2.20269142,-0.84260738,-0.01861448,0.711684614,-1.0814076,0.492742898,0.09633553,-0.41465127,-1.0573779,0.4792118,⋯,-0.5531416,1.12677198,-0.6887637,0.07516654,-1.1501492,0.08811665,-1.6427782,-0.6124164,-0.0492491,-2.0683459


In [4]:
# Pick a cell type
cell_types <- unique(info$cluster)
acinar_labels <- cell_types[str_detect(cell_types,'Acinar')]
acinar_labels

[1] "Acinar_1_2_6" "Acinar_5"     "Acinar_3"     "Acinar_4"

In [5]:
#Filter devscores by cell type
ct_info <- info[info$cluster %in% acinar_labels,]
ct_info$cluster_sample <- paste0(ct_info$cluster, "::", ct_info$samples)
cell_type_barcodes <- rownames(ct_info)

ct_devscores <- devscores[cell_type_barcodes]
head(ct_devscores)

,MM_339_AAACGAATCCCGAAGC-1,MM_339_AAACGAATCTATGAGC-1,MM_339_AAACTCGGTGTGCTTA-1,MM_339_AAAGATGTCGGATGTT-1,MM_339_AAAGGATAGGAGTCTG-1,MM_339_AAAGGATAGTCGAGCA-1,MM_339_AAAGGGCCAGTCAGCC-1,MM_339_AAAGGGCTCGATATGC-1,MM_339_AAAGGGCTCGGTTAGT-1,MM_339_AAATGAGGTGCCAAGA-1,⋯,MM_343_TGATCAGAGTCCGTGC-1,MM_344_GGGTTATGTGGTCGAA-1,MM_344_TTACTTGTCACGATTG-1,MM_365_AGGCGTCGTGCAAAGC-1,MM_365_TTGTCTATCCATAACG-1,MM_388_AGGACGATCGGTCTCT-1,MM_392_GTCGTAAGTTAACTCG-1,MM_394_TAAGTGCCAGGCTACC-1,MM_543_TGCTATTAGTCAACTC-1,MM_545_GCCATAATCGATGAAA-1
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MA0030.1_FOXF2,0.54520151,0.40990183,3.35607428,2.000801239,0.9658235,-0.459021816,0.44334913,-0.25558497,-1.6471011,0.8835954,⋯,0.11774177,0.05098749,-0.04810347,0.3004695,2.24742975,0.16995892,0.91933604,0.6541884,0.545410549,-0.2933488
MA0031.1_FOXD1,-0.64639620,0.56733851,2.16471443,0.222432873,0.1040411,-1.840425687,1.07262678,-0.05391174,1.3205662,0.8736271,⋯,-0.11478954,-1.99743894,0.95563125,1.5885369,-0.38897092,-0.01416535,0.12739287,0.4347822,-1.319355763,1.6584833
MA0051.1_IRF2,-0.04734576,0.21278816,-1.17043162,0.006864465,-0.6147434,0.779877039,-0.95418391,0.84427104,-0.1533793,-0.6127317,⋯,-0.41060749,-2.34039655,1.96020891,0.9460906,-1.19794304,-2.58492544,-0.07967277,0.8009769,0.939301684,0.9750772
MA0059.1_MAX::MYC,-0.01129497,0.06407827,-0.60875582,-0.568995229,1.0334839,0.008193637,-1.30026440,-0.93085128,-2.0120367,-0.2730295,⋯,-0.74872369,0.19077289,1.77105142,-0.3716778,-0.29008807,-0.91362941,-0.51691895,0.7709466,0.045776489,0.2942622
MA0066.1_PPARG,0.51365043,-0.50851101,-0.55188880,-0.488296947,-0.6396850,0.131552195,-0.30285472,1.31740292,1.8427194,0.4069058,⋯,1.57030957,0.30522468,0.16772170,0.2058686,0.02586427,-0.29000089,-1.41702940,-1.4548441,-0.007677915,0.7754606
MA0069.1_PAX6,-2.20269142,-0.84260738,-0.01861448,0.711684614,-1.0814076,0.492742898,0.09633553,-0.41465127,-1.0573779,0.4792118,⋯,0.03741945,-0.55272681,0.01524922,-2.9344526,-0.22372915,-2.11376888,-0.53025330,-1.1320367,-0.996974986,-2.0408134


# Significance Testing

Asked to try all versus one t-tests 7/20/23

In [6]:
#Check the cluster labels
acinar_labels

[1] "Acinar_1_2_6" "Acinar_5"     "Acinar_3"     "Acinar_4"

In [7]:
# Average motif accessibiltiy by cluster and sample
clust_samp = sapply(unique(ct_info$cluster_sample), 
            function(i) rowMeans(ct_devscores[,ct_info$cells[ct_info$cluster_sample==i],drop = FALSE], na.rm=T))

clust_samp
                    
clust_samp <- data.frame(t(clust_samp), check.names=FALSE)
                
clust_samp$cluster <- str_split(rownames(clust_samp), "::", simplify=T)[,1]
clust_samp$sample <- str_split(rownames(clust_samp), "::", simplify=T)[,2]
                    
dim(clust_samp)
head(clust_samp)

,Acinar_1_2_6::6247,Acinar_5::6247,Acinar_3::6247,Acinar_4::6247,Acinar_1_2_6::6418,Acinar_3::6418,Acinar_1_2_6::6339,Acinar_3::6339,Acinar_4::6339,Acinar_1_2_6::6197,⋯,Acinar_4::6512,Acinar_5::6505,Acinar_4::6505,Acinar_5::6339,Acinar_5::6310,Acinar_4::6429,Acinar_4::6310,Acinar_4::6418,Acinar_4::6401,Acinar_4::6380
MA0030.1_FOXF2,0.288221390,0.911368329,0.67485029,0.94419791,0.062074954,0.203942062,1.196746e-01,0.449885917,0.56658238,0.21025019,⋯,-0.37281217,0.58291194,-0.68341087,0.56915239,-0.37969559,1.092085042,0.9998722313,-0.5625307,-0.83559611,1.37100768
MA0031.1_FOXD1,0.343491514,0.925559109,0.82597127,0.76108221,0.109934165,0.288840758,2.365999e-01,0.719279209,1.38852639,0.24758389,⋯,0.87443622,0.42817360,0.65668678,0.91200934,0.70278856,0.852518202,-0.4133845749,-0.6824695,0.66822500,1.22846203
MA0051.1_IRF2,-0.343361258,-0.941498166,-0.63863696,-1.21497578,-0.095611360,-0.203231519,-1.629510e-01,-0.358621929,-0.34783746,-0.39496448,⋯,-0.56390143,-0.37800792,0.20202006,-0.78279488,-0.63233232,-0.734522389,-0.0472231470,-0.3442975,0.27620688,-1.86185353
MA0059.1_MAX::MYC,-0.022240786,-0.119718916,-0.18223376,-0.46287207,0.001444589,-0.096570137,6.589320e-02,0.181919273,-0.30285498,-0.03177845,⋯,-0.13714213,-0.36040976,0.10278185,-0.18874351,0.89579583,0.213606246,-0.4265636351,-0.3221896,0.15045274,0.06719507
MA0066.1_PPARG,0.002049667,0.090518460,-0.01236141,-0.13239338,0.037376184,0.094626248,3.951064e-02,0.028633112,0.01630102,0.01085940,⋯,0.78459220,0.34669837,0.21038560,-0.05416728,0.59205959,0.563816043,0.4178790450,0.6175109,-1.79289471,1.14990643
MA0069.1_PAX6,-0.112726661,-0.414933019,-0.26612253,-0.56866442,-0.058659786,-0.046901089,-3.161382e-02,-0.165541586,-1.21446828,-0.16841572,⋯,-0.69352349,-0.31982734,-0.69850063,0.03471583,-1.05245416,-0.899182816,-1.4203301898,-0.7835081,1.07087680,0.61136522
MA0070.1_PBX1,0.075011763,0.448681793,0.09432914,0.01501102,0.034099768,0.061751548,-7.907521e-03,0.151451677,-0.32864573,0.03629012,⋯,0.24118900,-0.01364684,1.37645127,-0.29529326,1.16896878,0.531887420,0.0018618467,-0.1977556,0.04108113,0.30749275
MA0071.1_RORA,0.109589020,0.381285074,0.20488259,0.24476057,0.014143555,0.054371139,2.768817e-02,0.092994990,0.75401909,0.18508580,⋯,0.30496000,0.07787003,-0.24759140,0.35917895,0.83390413,1.187132462,0.2601523977,0.1148058,-1.06967332,-1.21552848
MA0072.1_RORA,-0.057396369,-0.281455428,-0.17533705,-0.21475368,-0.072812574,-0.179599266,-4.940730e-02,-0.148294930,-1.25190180,-0.06546575,⋯,-0.17027318,-0.15668805,0.12711748,0.04504954,0.52084069,0.385848087,0.0009031315,-1.2135482,-1.75507529,-0.77371536
MA0073.1_RREB1,-0.180194181,-0.505641784,-0.35210531,-0.27179953,-0.081031600,-0.171604721,-6.438620e-02,-0.337941823,-0.07808865,-0.24790279,⋯,-0.67090991,-0.57909444,-0.57229097,-0.04004149,-0.90249966,-0.706140507,-0.6816572883,-0.5063685,-0.48188916,-0.67495108


[1] 115 694

,MA0030.1_FOXF2,MA0031.1_FOXD1,MA0051.1_IRF2,MA0059.1_MAX::MYC,MA0066.1_PPARG,MA0069.1_PAX6,MA0070.1_PBX1,MA0071.1_RORA,MA0072.1_RORA,MA0073.1_RREB1,⋯,MA1532.2_NR1D2,MA1547.2_PITX2,MA1563.2_SOX18,MA1566.2_TBX3,MA1601.2_ZNF75D,MA1630.2_ZNF281,MA1633.2_BACH1,MA0597.2_THAP1,cluster,sample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
Acinar_1_2_6::6247,0.28822139,0.3434915,-0.34336126,-0.022240786,0.002049667,-0.11272666,0.07501176,0.10958902,-0.05739637,-0.1801942,⋯,-0.043750999,0.05277552,0.08216360,0.4942917,0.107007894,-0.2421891,-0.9596892,0.16714869,Acinar_1_2_6,6247
Acinar_5::6247,0.91136833,0.9255591,-0.94149817,-0.119718916,0.090518460,-0.41493302,0.44868179,0.38128507,-0.28145543,-0.5056418,⋯,-0.206643097,0.22356241,0.14820043,1.5264949,0.285092066,-0.8554300,-2.1194473,0.53734369,Acinar_5,6247
Acinar_3::6247,0.67485029,0.8259713,-0.63863696,-0.182233762,-0.012361412,-0.26612253,0.09432914,0.20488259,-0.17533705,-0.3521053,⋯,0.042034870,0.06211239,0.11476671,0.9643146,0.165063201,-0.5019049,-1.2343873,0.32778600,Acinar_3,6247
Acinar_4::6247,0.94419791,0.7610822,-1.21497578,-0.462872068,-0.132393377,-0.56866442,0.01501102,0.24476057,-0.21475368,-0.2717995,⋯,0.213682681,0.04341841,0.20598252,1.6425372,0.388253213,-0.7714226,-3.0677926,0.55890092,Acinar_4,6247
Acinar_1_2_6::6418,0.06207495,0.1099342,-0.09561136,0.001444589,0.037376184,-0.05865979,0.03409977,0.01414355,-0.07281257,-0.0810316,⋯,0.005201858,0.03719241,0.03668276,0.1519975,0.001158809,-0.1024669,0.0801505,0.08782445,Acinar_1_2_6,6418
Acinar_3::6418,0.20394206,0.2888408,-0.20323152,-0.096570137,0.094626248,-0.04690109,0.06175155,0.05437114,-0.17959927,-0.1716047,⋯,0.017403146,0.06063815,0.05915057,0.4625351,0.038456187,-0.2675104,0.5915009,0.16260266,Acinar_3,6418


In [8]:
# Run an anova comparing the motifs accessibility across clusters, controlling for sample
clust_anova_data <- select(clust_samp, samples=sample, cluster, everything())
results <- numeric(0)

donor_anova_results = data.frame(matrix(nrow=nrow(ct_devscores), ncol = 2))
rownames(donor_anova_results)= colnames(clust_anova_data)[3:ncol(clust_anova_data)]
    
#for (i in 3) {
for (i in 3:ncol(clust_anova_data)) {
    motif <- colnames(clust_anova_data)[i]
    mod1 = aov(clust_anova_data[,i] ~ cluster + samples, data = clust_anova_data)
    res <- Anova(mod1, type="II")
    
    donor_anova_results[motif,1] <- motif
    donor_anova_results[motif,2] <- as.numeric(broom::tidy(res)[1,5])
}

dim(donor_anova_results)
head(donor_anova_results)

[1] 692   2

,X1,X2
,<chr>,<dbl>
MA0030.1_FOXF2,MA0030.1_FOXF2,6.959097e-04
MA0031.1_FOXD1,MA0031.1_FOXD1,1.307808e-11
MA0051.1_IRF2,MA0051.1_IRF2,1.554499e-14
MA0059.1_MAX::MYC,MA0059.1_MAX::MYC,1.119976e-02
MA0066.1_PPARG,MA0066.1_PPARG,6.430220e-01
MA0069.1_PAX6,MA0069.1_PAX6,5.950115e-06


In [9]:
#FDR Correct and check results
colnames(donor_anova_results)[1:2] <- c('Motif','p.value')
donor_anova_results <- arrange(donor_anova_results, p.value)
donor_anova_results$q.values <- qvalue(donor_anova_results$p.value)$qvalue

dim(donor_anova_results)
sum(donor_anova_results$p.value < 0.05)
sum(donor_anova_results$q.value < .05)
head(donor_anova_results)
tail(donor_anova_results)

[1] 692   3

[1] 534

[1] 610

,Motif,p.value,q.values
,<chr>,<dbl>,<dbl>
1,MA0745.2_SNAI2,3.368285e-42,3.073344e-40
2,MA0103.3_ZEB1,2.124091e-41,9.690483e-40
3,MA0499.2_MYOD1,2.640159e-38,8.029921e-37
4,MA1559.1_SNAI3,1.195929e-37,2.399726e-36
5,MA1935.1_ERF::FOXI1,1.315011e-37,2.399726e-36
6,MA0522.3_TCF3,5.413610e-37,7.642365e-36


,Motif,p.value,q.values
,<chr>,<dbl>,<dbl>
687,MA1644.1_NFYC,0.9412758,0.1250151
688,MA2003.1_NKX2-4,0.9526814,0.1263460
689,MA1474.1_CREB3L4,0.9546158,0.1264188
690,MA0794.1_PROX1,0.9646204,0.1275586
691,MA0865.2_E2F8,0.9855639,0.1301395
692,MA1645.1_NKX2-2,0.9991224,0.1317392


In [10]:
#Post hoc compare individual clusters to everything else and correct with holm's

type_anova_results_full = data.frame(matrix(nrow=nrow(ct_devscores), ncol = 1))
colnames(type_anova_results_full) <- c('Motif')
type_anova_results_full$Motif <- rownames(ct_devscores)

for (type in acinar_labels) {
    type_anova_data <- select(clust_samp, samples=sample, cluster, everything())
    
    type_anova_data$cluster <- type_anova_data$cluster ==type
    type_anova_data <- group_by(type_anova_data, cluster, samples) %>%
                            summarize_all(mean)
    
    type_anova_results = data.frame(matrix(nrow=nrow(ct_devscores), ncol = 2))
    colnames(type_anova_results) <- c('Motif',paste0(type, ".p.value"))
    rownames(type_anova_results)= colnames(type_anova_data)[3:ncol(type_anova_data)]
        
    #for (i in 3) {
    for (i in 3:ncol(type_anova_data)) {
        motif <- colnames(type_anova_data)[i]
        mod1 = aov(unlist(type_anova_data[,i]) ~ cluster + samples, data = type_anova_data)
        res <- Anova(mod1, type="II")
        
        type_anova_results[motif,'Motif'] <- motif
        type_anova_results[motif,2] <- as.numeric(broom::tidy(res)[1,5])
    }
        
    type_anova_results_full <- left_join(type_anova_results_full, type_anova_results)
}

dim(type_anova_results_full)
head(type_anova_results_full)

Joining with `by = join_by(Motif)`
Joining with `by = join_by(Motif)`
Joining with `by = join_by(Motif)`
Joining with `by = join_by(Motif)`


[1] 692   5

,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,MA0030.1_FOXF2,1.878561e-06,7.161564e-03,0.4033856373,0.4695431201
2,MA0031.1_FOXD1,8.190823e-11,2.474089e-04,0.7115260632,0.0071438760
3,MA0051.1_IRF2,2.146152e-10,1.182946e-07,0.0001289029,0.0008358103
4,MA0059.1_MAX::MYC,7.680233e-03,1.116966e-01,0.0013265279,0.1502414143
5,MA0066.1_PPARG,3.355028e-01,2.115934e-01,0.2765041439,0.6371728416
6,MA0069.1_PAX6,1.016753e-06,6.478739e-03,0.0012452074,0.0280443027


In [11]:
#Run the holm's correction
holm_df <- data.frame()

for (i in 1:nrow(type_anova_results_full)) {
    holm_adjust <- p.adjust(type_anova_results_full[i,2:5], method='holm')
    names(holm_adjust) <- str_replace(names(holm_adjust), 'p.value','holm')
    holm_adjust <- data.frame(t(data.frame(holm_adjust)))
    holm_adjust$Motif <- type_anova_results_full[i,1]
    
    holm_df <- rbind(holm_df, data.frame(holm_adjust))
}

type_anova_results_full_holm <- left_join(type_anova_results_full, holm_df)

dim(type_anova_results_full_holm)
sum(is.na(type_anova_results_full_holm))
head(type_anova_results_full_holm)

Joining with `by = join_by(Motif)`


[1] 692   9

[1] 0

,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,MA0030.1_FOXF2,1.878561e-06,7.161564e-03,0.4033856373,0.4695431201,7.514245e-06,2.148469e-02,0.8067712747,0.8067712747
2,MA0031.1_FOXD1,8.190823e-11,2.474089e-04,0.7115260632,0.0071438760,3.276329e-10,7.422268e-04,0.7115260632,0.0142877519
3,MA0051.1_IRF2,2.146152e-10,1.182946e-07,0.0001289029,0.0008358103,8.584606e-10,3.548837e-07,0.0002578058,0.0008358103
4,MA0059.1_MAX::MYC,7.680233e-03,1.116966e-01,0.0013265279,0.1502414143,2.304070e-02,2.233933e-01,0.0053061117,0.2233932673
5,MA0066.1_PPARG,3.355028e-01,2.115934e-01,0.2765041439,0.6371728416,8.463736e-01,8.463736e-01,0.8463735940,0.8463735940
6,MA0069.1_PAX6,1.016753e-06,6.478739e-03,0.0012452074,0.0280443027,4.067012e-06,1.295748e-02,0.0037356223,0.0280443027


In [12]:
# Join with the overall ANOVA data
colnames(donor_anova_results)[str_detect(colnames(donor_anova_results), 'p.value')] <- 'anova.p.value'
colnames(donor_anova_results)[str_detect(colnames(donor_anova_results), 'q.value')] <- 'anova.q.value'
#donor_anova_results
type_anova_results_full_holm_anova <- left_join(type_anova_results_full_holm, donor_anova_results)

dim(type_anova_results_full_holm_anova)
head(type_anova_results_full_holm_anova)

Joining with `by = join_by(Motif)`


[1] 692  11

,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,anova.q.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,MA0030.1_FOXF2,1.878561e-06,7.161564e-03,0.4033856373,0.4695431201,7.514245e-06,2.148469e-02,0.8067712747,0.8067712747,6.959097e-04,1.515449e-04
2,MA0031.1_FOXD1,8.190823e-11,2.474089e-04,0.7115260632,0.0071438760,3.276329e-10,7.422268e-04,0.7115260632,0.0142877519,1.307808e-11,5.709526e-12
3,MA0051.1_IRF2,2.146152e-10,1.182946e-07,0.0001289029,0.0008358103,8.584606e-10,3.548837e-07,0.0002578058,0.0008358103,1.554499e-14,8.294624e-15
4,MA0059.1_MAX::MYC,7.680233e-03,1.116966e-01,0.0013265279,0.1502414143,2.304070e-02,2.233933e-01,0.0053061117,0.2233932673,1.119976e-02,2.089788e-03
5,MA0066.1_PPARG,3.355028e-01,2.115934e-01,0.2765041439,0.6371728416,8.463736e-01,8.463736e-01,0.8463735940,0.8463735940,6.430220e-01,8.943847e-02
6,MA0069.1_PAX6,1.016753e-06,6.478739e-03,0.0012452074,0.0280443027,4.067012e-06,1.295748e-02,0.0037356223,0.0280443027,5.950115e-06,1.635271e-06


In [13]:
# Calculate deviation score average for different comparisons
dev_averages <- data.frame(Motif=colnames(clust_samp)[1:(ncol(clust_samp) - 2)])

for (clust in acinar_labels) {
    cluster_specific <- clust_samp
    cluster_specific$cluster <- cluster_specific$cluster == clust
    
    cluster_specific <- group_by(cluster_specific, cluster, sample) %>%
        summarise_all(mean) %>%
        select(-sample) %>%
        group_by(cluster) %>%
        summarise_all(mean)
    
    cluster_specific$cluster[cluster_specific$cluster] <- clust
    cluster_specific$cluster[cluster_specific$cluster == FALSE] <- paste0("not.",clust)
    
    cluster_specific <- tibble::column_to_rownames(cluster_specific, var='cluster')
    cluster_specific <- data.frame(t(cluster_specific))
    cluster_specific <- tibble::rownames_to_column(cluster_specific, var='Motif')
    
    cluster_specific[[paste0(clust, '.diff')]] <- cluster_specific[[clust]] - cluster_specific[[paste0("not.",clust)]]
    
    dev_averages <- left_join(dev_averages, cluster_specific)
}

Joining with `by = join_by(Motif)`
Joining with `by = join_by(Motif)`
Joining with `by = join_by(Motif)`
Joining with `by = join_by(Motif)`


In [14]:
# Join in deviation scores
type_anova_results_full_holm_anova_dev <- left_join(type_anova_results_full_holm_anova, dev_averages)

type_anova_results_full_holm_anova_dev

Joining with `by = join_by(Motif)`


Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,Acinar_1_2_6.diff,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MA0030.1_FOXF2,1.878561e-06,7.161564e-03,4.033856e-01,4.695431e-01,7.514245e-06,2.148469e-02,8.067713e-01,8.067713e-01,6.959097e-04,⋯,-0.29368123,0.33699151,0.51742727,0.18043576,0.37565291,0.40839287,0.032739964,0.36346805,0.449233452,0.08576540
MA0031.1_FOXD1,8.190823e-11,2.474089e-04,7.115261e-01,7.143876e-03,3.276329e-10,7.422268e-04,7.115261e-01,1.428775e-02,1.307808e-11,⋯,-0.47908016,0.51777387,0.75361113,0.23583726,0.58349083,0.57171637,-0.011774461,0.51572609,0.780814962,0.26508887
MA0051.1_IRF2,2.146152e-10,1.182946e-07,1.289029e-04,8.358103e-04,8.584606e-10,3.548837e-07,2.578058e-04,8.358103e-04,1.554499e-14,⋯,0.42732160,-0.45760182,-0.77134101,-0.31373919,-0.58512056,-0.40493006,0.180190501,-0.46521052,-0.773251983,-0.30804147
MA0059.1_MAX::MYC,7.680233e-03,1.116966e-01,1.326528e-03,1.502414e-01,2.304070e-02,2.233933e-01,5.306112e-03,2.233933e-01,1.119976e-02,⋯,-0.08194305,0.06217336,0.15690768,0.09473432,0.11512127,0.01296847,-0.102152791,0.06592540,0.151213805,0.08528841
MA0066.1_PPARG,3.355028e-01,2.115934e-01,2.765041e-01,6.371728e-01,8.463736e-01,8.463736e-01,8.463736e-01,8.463736e-01,6.430220e-01,⋯,-0.04471532,0.04754416,0.09334931,0.04580514,0.04802604,0.09448791,0.046461871,0.07143713,0.019434045,-0.05200309
MA0069.1_PAX6,1.016753e-06,6.478739e-03,1.245207e-03,2.804430e-02,4.067012e-06,1.295748e-02,3.735622e-03,2.804430e-02,5.950115e-06,⋯,0.23944826,-0.24760523,-0.39473373,-0.14712850,-0.31250140,-0.20176058,0.110740822,-0.23391655,-0.451613101,-0.21769655
MA0070.1_PBX1,1.950325e-04,1.071945e-01,5.157297e-02,2.224225e-01,7.801299e-04,2.143890e-01,1.547189e-01,2.224225e-01,1.918401e-02,⋯,-0.12903260,0.11429194,0.22582676,0.11153482,0.16187921,0.08662911,-0.075250100,0.11966978,0.215342409,0.09567262
MA0071.1_RORA,1.189834e-06,1.795454e-04,9.240308e-01,4.904732e-01,4.759337e-06,5.386362e-04,9.809463e-01,9.809463e-01,2.142472e-04,⋯,-0.24277745,0.26911233,0.44937995,0.18026762,0.31537092,0.31243019,-0.002940737,0.29809079,0.371014208,0.07292341
MA0072.1_RORA,2.290104e-02,5.710456e-01,8.850272e-01,2.716860e-01,9.160417e-02,1.000000e+00,1.000000e+00,8.150579e-01,2.645968e-01,⋯,0.08079290,-0.09834624,-0.06669427,0.03165197,-0.08887440,-0.09365701,-0.004782615,-0.06330595,-0.176956520,-0.11365057


In [15]:
# Determine which cluter the motif is most accessible in
maxes <- apply(X=type_anova_results_full_holm_anova_dev[acinar_labels], MARGIN=1, FUN=max)

maxes_labels <- maxes

maxes_labels[maxes_labels==type_anova_results_full_holm_anova_dev$Acinar_1_2_6] <- 'Acinar_1_2_6'
maxes_labels[maxes_labels==type_anova_results_full_holm_anova_dev$Acinar_3] <- 'Acinar_3'
maxes_labels[maxes_labels==type_anova_results_full_holm_anova_dev$Acinar_4] <- 'Acinar_4'
maxes_labels[maxes_labels==type_anova_results_full_holm_anova_dev$Acinar_5] <- 'Acinar_5'

#maxes_labels

type_anova_results_full_holm_anova_dev$max <- maxes_labels

type_anova_results_full_holm_anova_dev

Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
MA0030.1_FOXF2,1.878561e-06,7.161564e-03,4.033856e-01,4.695431e-01,7.514245e-06,2.148469e-02,8.067713e-01,8.067713e-01,6.959097e-04,⋯,0.33699151,0.51742727,0.18043576,0.37565291,0.40839287,0.032739964,0.36346805,0.449233452,0.08576540,Acinar_5
MA0031.1_FOXD1,8.190823e-11,2.474089e-04,7.115261e-01,7.143876e-03,3.276329e-10,7.422268e-04,7.115261e-01,1.428775e-02,1.307808e-11,⋯,0.51777387,0.75361113,0.23583726,0.58349083,0.57171637,-0.011774461,0.51572609,0.780814962,0.26508887,Acinar_4
MA0051.1_IRF2,2.146152e-10,1.182946e-07,1.289029e-04,8.358103e-04,8.584606e-10,3.548837e-07,2.578058e-04,8.358103e-04,1.554499e-14,⋯,-0.45760182,-0.77134101,-0.31373919,-0.58512056,-0.40493006,0.180190501,-0.46521052,-0.773251983,-0.30804147,Acinar_1_2_6
MA0059.1_MAX::MYC,7.680233e-03,1.116966e-01,1.326528e-03,1.502414e-01,2.304070e-02,2.233933e-01,5.306112e-03,2.233933e-01,1.119976e-02,⋯,0.06217336,0.15690768,0.09473432,0.11512127,0.01296847,-0.102152791,0.06592540,0.151213805,0.08528841,Acinar_5
MA0066.1_PPARG,3.355028e-01,2.115934e-01,2.765041e-01,6.371728e-01,8.463736e-01,8.463736e-01,8.463736e-01,8.463736e-01,6.430220e-01,⋯,0.04754416,0.09334931,0.04580514,0.04802604,0.09448791,0.046461871,0.07143713,0.019434045,-0.05200309,Acinar_3
MA0069.1_PAX6,1.016753e-06,6.478739e-03,1.245207e-03,2.804430e-02,4.067012e-06,1.295748e-02,3.735622e-03,2.804430e-02,5.950115e-06,⋯,-0.24760523,-0.39473373,-0.14712850,-0.31250140,-0.20176058,0.110740822,-0.23391655,-0.451613101,-0.21769655,Acinar_1_2_6
MA0070.1_PBX1,1.950325e-04,1.071945e-01,5.157297e-02,2.224225e-01,7.801299e-04,2.143890e-01,1.547189e-01,2.224225e-01,1.918401e-02,⋯,0.11429194,0.22582676,0.11153482,0.16187921,0.08662911,-0.075250100,0.11966978,0.215342409,0.09567262,Acinar_5
MA0071.1_RORA,1.189834e-06,1.795454e-04,9.240308e-01,4.904732e-01,4.759337e-06,5.386362e-04,9.809463e-01,9.809463e-01,2.142472e-04,⋯,0.26911233,0.44937995,0.18026762,0.31537092,0.31243019,-0.002940737,0.29809079,0.371014208,0.07292341,Acinar_5
MA0072.1_RORA,2.290104e-02,5.710456e-01,8.850272e-01,2.716860e-01,9.160417e-02,1.000000e+00,1.000000e+00,8.150579e-01,2.645968e-01,⋯,-0.09834624,-0.06669427,0.03165197,-0.08887440,-0.09365701,-0.004782615,-0.06330595,-0.176956520,-0.11365057,Acinar_1_2_6


In [16]:
# Number of Motifs significant from holm's and overall anova
paste0("Acinar_basal motifs passing holms & anova FDR: ", sum(type_anova_results_full_holm_anova_dev$Acinar_1_2_6.holm < 0.05 & type_anova_results_full_holm_anova_dev$anova.q.value < .05))
paste0("Acinar_signaling_1 motifs passing holms & anova FDR: ", sum(type_anova_results_full_holm_anova_dev$Acinar_3.holm < 0.05 & type_anova_results_full_holm_anova_dev$anova.q.value < .05))
paste0("Acinar_secreting motifs passing holms & anova FDR: ", sum(type_anova_results_full_holm_anova_dev$Acinar_4.holm < 0.05 & type_anova_results_full_holm_anova_dev$anova.q.value < .05))
paste0("Acinar_signaling_2 motifs passing holms & anova FDR: ", sum(type_anova_results_full_holm_anova_dev$Acinar_5.holm < 0.05 & type_anova_results_full_holm_anova_dev$anova.q.value < .05))

[1] "Acinar_basal motifs passing holms & anova FDR: 523"

[1] "Acinar_signaling_1 motifs passing holms & anova FDR: 343"

[1] "Acinar_secreting motifs passing holms & anova FDR: 345"

[1] "Acinar_signaling_2 motifs passing holms & anova FDR: 349"

In [17]:
# Number of Motifs significant from holm's and overall anova and positively enriched
paste0("Acinar_basal motifs passing holms & anova FDR & positively enriched: ", 
       sum(type_anova_results_full_holm_anova_dev$Acinar_1_2_6.holm < 0.05 & 
           type_anova_results_full_holm_anova_dev$anova.q.value < .05 & 
           type_anova_results_full_holm_anova_dev$Acinar_1_2_6.diff > 0))
paste0("Acinar_signaling_1 motifs passing holms & anova FDR & positively enriched: ", 
       sum(type_anova_results_full_holm_anova_dev$Acinar_3.holm < 0.05 & 
           type_anova_results_full_holm_anova_dev$anova.q.value < .05 & 
           type_anova_results_full_holm_anova_dev$Acinar_3.diff > 0))
paste0("Acinar_secreting motifs passing holms & anova FDR & positively enriched: ", 
       sum(type_anova_results_full_holm_anova_dev$Acinar_4.holm < 0.05 & 
           type_anova_results_full_holm_anova_dev$anova.q.value < .05 & 
           type_anova_results_full_holm_anova_dev$Acinar_4.diff > 0))
paste0("Acinar_signaling_2 motifs passing holms & anova FDR & positively enriched: ", 
       sum(type_anova_results_full_holm_anova_dev$Acinar_5.holm < 0.05 & 
           type_anova_results_full_holm_anova_dev$anova.q.value < .05 & 
           type_anova_results_full_holm_anova_dev$Acinar_5.diff > 0))

[1] "Acinar_basal motifs passing holms & anova FDR & positively enriched: 334"

[1] "Acinar_signaling_1 motifs passing holms & anova FDR & positively enriched: 221"

[1] "Acinar_secreting motifs passing holms & anova FDR & positively enriched: 127"

[1] "Acinar_signaling_2 motifs passing holms & anova FDR & positively enriched: 127"

In [18]:
# Number of Motifs significant from holm's and overall anova and max deviation score in a given cluster
paste0("Acinar_basal motifs passing holms & anova FDR & greatest in subtype: ", 
       sum(type_anova_results_full_holm_anova_dev$Acinar_1_2_6.holm < 0.05 & 
           type_anova_results_full_holm_anova_dev$anova.q.value < .05 & 
           type_anova_results_full_holm_anova_dev$max == 'Acinar_1_2_6'))
paste0("Acinar_signaling_1 motifs passing holms & anova FDR & greatest in subtype: ", 
       sum(type_anova_results_full_holm_anova_dev$Acinar_3.holm < 0.05 & 
           type_anova_results_full_holm_anova_dev$anova.q.value < .05 & 
           type_anova_results_full_holm_anova_dev$max == 'Acinar_3'))
paste0("Acinar_secreting motifs passing holms & anova FDR & greatest in subtype: ", 
       sum(type_anova_results_full_holm_anova_dev$Acinar_4.holm < 0.05 & 
           type_anova_results_full_holm_anova_dev$anova.q.value < .05 & 
           type_anova_results_full_holm_anova_dev$max == 'Acinar_4'))
paste0("Acinar_signaling_2 motifs passing holms & anova FDR & greatest in subtype: ", 
       sum(type_anova_results_full_holm_anova_dev$Acinar_5.holm < 0.05 & 
           type_anova_results_full_holm_anova_dev$anova.q.value < .05 & 
           type_anova_results_full_holm_anova_dev$max == 'Acinar_5'))

[1] "Acinar_basal motifs passing holms & anova FDR & greatest in subtype: 295"

[1] "Acinar_signaling_1 motifs passing holms & anova FDR & greatest in subtype: 53"

[1] "Acinar_secreting motifs passing holms & anova FDR & greatest in subtype: 115"

[1] "Acinar_signaling_2 motifs passing holms & anova FDR & greatest in subtype: 40"

In [19]:
# Top hits by holm's significance
head(arrange(type_anova_results_full_holm_anova_dev, Acinar_1_2_6.holm), n=15)
head(arrange(type_anova_results_full_holm_anova_dev, Acinar_3.holm), n=15)
head(arrange(type_anova_results_full_holm_anova_dev, Acinar_4.holm), n=15)
head(arrange(type_anova_results_full_holm_anova_dev, Acinar_5.holm), n=15)

,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,MA0592.3_ESRRA,7.291057e-20,4.999179e-09,4.963146e-10,1.192519e-11,2.916423e-19,4.999179e-09,9.926293e-10,3.577556e-11,1.177175e-36,⋯,1.4851467,2.1205658,0.6354192,1.7271209,1.4155946,-0.3115263,1.3936511,2.462370,1.0687186,Acinar_4
2,MA0522.3_TCF3,7.807799e-20,4.058820e-09,1.232217e-09,8.678717e-13,3.123120e-19,4.058820e-09,2.464434e-09,2.603615e-12,5.413610e-37,⋯,2.1415102,3.1208336,0.9793234,2.5592279,1.9062310,-0.6529969,1.9948947,3.666486,1.6715909,Acinar_4
3,MA1559.1_SNAI3,8.164321e-20,7.540437e-09,6.905179e-12,5.997740e-13,3.265728e-19,7.540437e-09,1.381036e-11,1.799322e-12,1.195929e-37,⋯,2.4584340,3.5791786,1.1207447,2.9600018,2.1244986,-0.8355032,2.2668351,4.279997,2.0131617,Acinar_4
4,MA0745.2_SNAI2,8.832115e-20,8.553561e-10,3.539203e-15,7.046577e-15,3.532846e-19,8.553561e-10,1.061761e-14,1.409315e-14,3.368285e-42,⋯,2.4043512,3.4802458,1.0758946,2.8764374,2.1061475,-0.7702898,2.2150132,4.169983,1.9549703,Acinar_4
5,MA1935.1_ERF::FOXI1,9.486416e-20,4.418449e-15,5.980277e-11,1.531661e-09,3.794566e-19,1.325535e-14,1.196055e-10,1.531661e-09,1.315011e-37,⋯,-1.5273116,-2.4015425,-0.8742309,-1.8522838,-1.4461984,0.4060854,-1.5193095,-2.492287,-0.9729771,Acinar_1_2_6
6,MA1558.1_SNAI1,1.223894e-19,4.951820e-09,2.315334e-11,3.311749e-12,4.895578e-19,4.951820e-09,4.630669e-11,9.935248e-12,1.292575e-36,⋯,2.3981229,3.5696414,1.1715185,2.9080490,2.1001217,-0.8079273,2.2490586,4.136897,1.8878384,Acinar_4
7,MA0103.3_ZEB1,1.574728e-19,2.689590e-10,2.472934e-10,3.423745e-15,6.298912e-19,4.945867e-10,4.945867e-10,1.027123e-14,2.124091e-41,⋯,2.3721703,3.4089431,1.0367728,2.8108584,2.1226393,-0.6882191,2.1942331,4.058333,1.8640995,Acinar_4
8,MA0762.1_ETV2,1.698016e-19,2.254607e-12,2.132797e-13,7.083345e-10,6.792064e-19,4.509213e-12,6.398392e-13,7.083345e-10,1.184151e-36,⋯,-1.1390021,-1.7888625,-0.6498604,-1.3888046,-1.0597060,0.3290985,-1.1240855,-1.884181,-0.7600950,Acinar_1_2_6
9,MA1648.1_TCF12,2.081354e-19,1.188093e-08,3.195115e-09,1.106954e-11,8.325415e-19,1.188093e-08,6.390230e-09,3.320862e-11,9.913070e-35,⋯,2.0172381,2.8897298,0.8724918,2.3942250,1.7880537,-0.6061713,1.8593418,3.463329,1.6039869,Acinar_4


,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,MA0745.2_SNAI2,8.832115e-20,8.553561e-10,3.539203e-15,7.046577e-15,3.532846e-19,8.553561e-10,1.061761e-14,1.409315e-14,3.368285e-42,⋯,2.4043512,3.4802458,1.0758946,2.8764374,2.10614753,-0.7702898,2.2150132,4.1699835,1.9549703,Acinar_4
2,MA0820.1_FIGLA,1.264630e-18,1.104855e-11,3.439739e-14,5.048727e-11,5.058522e-18,2.209710e-11,1.031922e-13,5.048727e-11,7.120192e-37,⋯,1.9833281,2.9388179,0.9554897,2.3893966,1.74871576,-0.6406808,1.8680009,3.3831063,1.5151054,Acinar_4
3,MA1529.1_NHLH2,1.749667e-15,2.003972e-07,6.815149e-14,4.192719e-12,6.998669e-15,2.003972e-07,2.044545e-13,8.385439e-12,4.587000e-33,⋯,0.6982516,1.0150484,0.3167969,0.8702506,0.51417624,-0.3560744,0.6088347,1.3248681,0.7160334,Acinar_4
4,MA0762.1_ETV2,1.698016e-19,2.254607e-12,2.132797e-13,7.083345e-10,6.792064e-19,4.509213e-12,6.398392e-13,7.083345e-10,1.184151e-36,⋯,-1.1390021,-1.7888625,-0.6498604,-1.3888046,-1.05970604,0.3290985,-1.1240855,-1.8841806,-0.7600950,Acinar_1_2_6
5,MA0499.2_MYOD1,7.702942e-18,6.171311e-12,6.802537e-13,1.712024e-12,3.081177e-17,6.171311e-12,2.040761e-12,3.424049e-12,2.640159e-38,⋯,1.7903112,2.6865896,0.8962785,2.1801882,1.53458054,-0.6456077,1.6744371,3.1255585,1.4511214,Acinar_4
6,MA1953.1_FOXO1::ELF1,2.578432e-17,3.474334e-12,3.390180e-12,3.360350e-10,1.031373e-16,1.017054e-11,1.017054e-11,3.360350e-10,6.465642e-35,⋯,-1.0508772,-1.6278010,-0.5769238,-1.2777848,-0.95141255,0.3263722,-1.0263564,-1.7479433,-0.7215869,Acinar_1_2_6
7,MA1559.1_SNAI3,8.164321e-20,7.540437e-09,6.905179e-12,5.997740e-13,3.265728e-19,7.540437e-09,1.381036e-11,1.799322e-12,1.195929e-37,⋯,2.4584340,3.5791786,1.1207447,2.9600018,2.12449856,-0.8355032,2.2668351,4.2799968,2.0131617,Acinar_4
8,MA1418.1_IRF3,3.088693e-14,2.587724e-11,9.863483e-12,1.723592e-05,1.235477e-13,5.175447e-11,2.959045e-11,1.723592e-05,2.274496e-24,⋯,-0.7470256,-1.2479890,-0.5009634,-0.9361842,-0.69175126,0.2444329,-0.7617550,-1.2389426,-0.4771876,Acinar_1_2_6
9,MA1950.1_FLI1::FOXI1,4.893387e-19,2.559704e-12,1.730355e-11,5.151576e-10,1.957355e-18,7.679112e-12,3.460709e-11,5.151576e-10,5.544068e-36,⋯,-1.4387260,-2.2222027,-0.7834767,-1.7408513,-1.33529374,0.4055575,-1.4096676,-2.3748369,-0.9651693,Acinar_1_2_6


,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,MA0103.3_ZEB1,1.574728e-19,2.689590e-10,2.472934e-10,3.423745e-15,6.298912e-19,4.945867e-10,4.945867e-10,1.027123e-14,2.124091e-41,⋯,2.3721703,3.408943,1.0367728,2.8108584,2.1226393,-0.6882191,2.1942331,4.058333,1.8640995,Acinar_4
2,MA0745.2_SNAI2,8.832115e-20,8.553561e-10,3.539203e-15,7.046577e-15,3.532846e-19,8.553561e-10,1.061761e-14,1.409315e-14,3.368285e-42,⋯,2.4043512,3.480246,1.0758946,2.8764374,2.1061475,-0.7702898,2.2150132,4.169983,1.9549703,Acinar_4
3,MA1631.1_ASCL1,4.326072e-19,1.965950e-09,9.486600e-10,5.352134e-13,1.730429e-18,1.965950e-09,1.897320e-09,1.605640e-12,5.863048e-37,⋯,2.2543412,3.330059,1.0757174,2.7142743,1.9933736,-0.7209008,2.1144846,3.861668,1.7471839,Acinar_4
4,MA1559.1_SNAI3,8.164321e-20,7.540437e-09,6.905179e-12,5.997740e-13,3.265728e-19,7.540437e-09,1.381036e-11,1.799322e-12,1.195929e-37,⋯,2.4584340,3.579179,1.1207447,2.9600018,2.1244986,-0.8355032,2.2668351,4.279997,2.0131617,Acinar_4
5,MA0522.3_TCF3,7.807799e-20,4.058820e-09,1.232217e-09,8.678717e-13,3.123120e-19,4.058820e-09,2.464434e-09,2.603615e-12,5.413610e-37,⋯,2.1415102,3.120834,0.9793234,2.5592279,1.9062310,-0.6529969,1.9948947,3.666486,1.6715909,Acinar_4
6,MA0499.2_MYOD1,7.702942e-18,6.171311e-12,6.802537e-13,1.712024e-12,3.081177e-17,6.171311e-12,2.040761e-12,3.424049e-12,2.640159e-38,⋯,1.7903112,2.686590,0.8962785,2.1801882,1.5345805,-0.6456077,1.6744371,3.125558,1.4511214,Acinar_4
7,MA0830.2_TCF4,3.758440e-18,4.265788e-08,1.979291e-10,2.728476e-12,1.503376e-17,4.265788e-08,3.958583e-10,8.185429e-12,1.743239e-34,⋯,2.4904315,3.620064,1.1296328,2.9775802,2.2015483,-0.7760320,2.3146526,4.270708,1.9560549,Acinar_4
8,MA1529.1_NHLH2,1.749667e-15,2.003972e-07,6.815149e-14,4.192719e-12,6.998669e-15,2.003972e-07,2.044545e-13,8.385439e-12,4.587000e-33,⋯,0.6982516,1.015048,0.3167969,0.8702506,0.5141762,-0.3560744,0.6088347,1.324868,0.7160334,Acinar_4
9,MA1558.1_SNAI1,1.223894e-19,4.951820e-09,2.315334e-11,3.311749e-12,4.895578e-19,4.951820e-09,4.630669e-11,9.935248e-12,1.292575e-36,⋯,2.3981229,3.569641,1.1715185,2.9080490,2.1001217,-0.8079273,2.2490586,4.136897,1.8878384,Acinar_4


,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,MA0081.2_SPIB,7.745088e-15,4.708318e-17,5.711467e-06,2.296461e-05,2.323526e-14,1.883327e-16,1.142293e-05,2.296461e-05,3.971125e-24,⋯,-1.734641,-2.756791,-1.0221504,-2.113147,-1.642111,0.4710355,-1.742295,-2.811179,-1.0688837,Acinar_1_2_6
2,MA1954.1_FOXO1::ELK1,1.199831e-16,1.545919e-15,7.279538e-10,1.743483e-07,4.799323e-16,4.637758e-15,1.455908e-09,1.743483e-07,3.328790e-31,⋯,-1.428903,-2.245363,-0.8164601,-1.732167,-1.346756,0.3854107,-1.423554,-2.324400,-0.9008462,Acinar_1_2_6
3,MA1935.1_ERF::FOXI1,9.486416e-20,4.418449e-15,5.980277e-11,1.531661e-09,3.794566e-19,1.325535e-14,1.196055e-10,1.531661e-09,1.315011e-37,⋯,-1.527312,-2.401542,-0.8742309,-1.852284,-1.446198,0.4060854,-1.519310,-2.492287,-0.9729771,Acinar_1_2_6
4,MA1508.1_IKZF1,1.050028e-15,5.935888e-15,5.974928e-07,1.185637e-06,4.200113e-15,1.780766e-14,1.194986e-06,1.194986e-06,1.072319e-27,⋯,-1.583073,-2.470274,-0.8872011,-1.910357,-1.503084,0.4072725,-1.574647,-2.567755,-0.9931076,Acinar_1_2_6
5,MA1952.1_FOXJ2::ELF1,1.086882e-17,2.621568e-14,3.537446e-06,1.151851e-06,4.347528e-17,7.864704e-14,3.537446e-06,2.303702e-06,6.058559e-28,⋯,-1.272090,-1.986184,-0.7140935,-1.535284,-1.204586,0.3306978,-1.261664,-2.074617,-0.8129535,Acinar_1_2_6
6,MA0761.2_ETV1,2.466881e-16,4.632721e-14,2.374780e-10,6.573434e-09,9.867525e-16,1.389816e-13,4.749559e-10,6.573434e-09,8.181550e-33,⋯,-1.732933,-2.737019,-1.0040861,-2.132874,-1.564195,0.5686792,-1.712402,-2.884182,-1.1717800,Acinar_1_2_6
7,MA1942.1_ETV2::FOXI1,4.382465e-18,4.751845e-14,9.723868e-11,7.725707e-09,1.752986e-17,1.425553e-13,1.944774e-10,7.725707e-09,2.047111e-34,⋯,-1.490005,-2.338224,-0.8482197,-1.809932,-1.395170,0.4147621,-1.480147,-2.435524,-0.9553772,Acinar_1_2_6
8,MA0598.3_EHF,1.450238e-14,6.469684e-14,1.074528e-08,7.947419e-08,5.800950e-14,1.940905e-13,2.149055e-08,7.947419e-08,7.109810e-29,⋯,-1.673239,-2.682426,-1.0091876,-2.079699,-1.487909,0.5917901,-1.659311,-2.806143,-1.1468318,Acinar_1_2_6
9,MA0062.3_GABPA,1.140049e-15,6.967626e-14,2.580405e-09,2.251794e-08,4.560197e-15,2.090288e-13,5.160810e-09,2.251794e-08,3.738038e-31,⋯,-1.745967,-2.788307,-1.0423396,-2.153691,-1.584873,0.5688180,-1.738586,-2.894696,-1.1561108,Acinar_1_2_6


In [20]:
# Most sig holm's motifs that are signicicant in overal anova and max deviation scores in the given cluster
head(filter(arrange(type_anova_results_full_holm_anova_dev, Acinar_1_2_6.holm), Acinar_1_2_6.holm < .05, anova.q.value < 0.05, max == 'Acinar_1_2_6'), n=15)
head(filter(arrange(type_anova_results_full_holm_anova_dev, Acinar_3.holm), Acinar_3.holm < .05, anova.q.value < 0.05, max == 'Acinar_3'), n=15)
head(filter(arrange(type_anova_results_full_holm_anova_dev, Acinar_4.holm), Acinar_4.holm < .05, anova.q.value < 0.05, max == 'Acinar_4'), n=15)
head(filter(arrange(type_anova_results_full_holm_anova_dev, Acinar_5.holm), Acinar_5.holm < .05, anova.q.value < 0.05, max == 'Acinar_5'), n=15)

,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,MA1935.1_ERF::FOXI1,9.486416e-20,4.418449e-15,5.980277e-11,1.531661e-09,3.794566e-19,1.325535e-14,1.196055e-10,1.531661e-09,1.315011e-37,⋯,-1.5273116,-2.4015425,-0.8742309,-1.8522838,-1.4461984,0.4060854,-1.5193095,-2.4922866,-0.9729771,Acinar_1_2_6
2,MA0762.1_ETV2,1.698016e-19,2.254607e-12,2.132797e-13,7.083345e-10,6.792064e-19,4.509213e-12,6.398392e-13,7.083345e-10,1.184151e-36,⋯,-1.1390021,-1.7888625,-0.6498604,-1.3888046,-1.0597060,0.3290985,-1.1240855,-1.8841806,-0.7600950,Acinar_1_2_6
3,MA1957.1_HOXB2::ELK1,4.588299e-19,1.687890e-05,1.219505e-03,1.694385e-07,1.835320e-18,3.375781e-05,1.219505e-03,5.083155e-07,3.902140e-22,⋯,-0.6419938,-0.9710495,-0.3290556,-0.7675258,-0.6149673,0.1525584,-0.6225182,-1.0612205,-0.4387023,Acinar_1_2_6
4,MA1950.1_FLI1::FOXI1,4.893387e-19,2.559704e-12,1.730355e-11,5.151576e-10,1.957355e-18,7.679112e-12,3.460709e-11,5.151576e-10,5.544068e-36,⋯,-1.4387260,-2.2222027,-0.7834767,-1.7408513,-1.3352937,0.4055575,-1.4096676,-2.3748369,-0.9651693,Acinar_1_2_6
5,MA0076.2_ELK4,4.948097e-19,7.062845e-09,9.144070e-08,1.123998e-10,1.979239e-18,1.412569e-08,9.144070e-08,3.371993e-10,1.088123e-32,⋯,-1.1258931,-1.6713416,-0.5454485,-1.3424968,-1.0395699,0.3029269,-1.0783878,-1.8653499,-0.7869621,Acinar_1_2_6
6,MA0475.2_FLI1,1.314674e-18,3.541591e-09,3.262911e-08,4.439355e-09,5.258697e-18,1.062477e-08,3.262911e-08,1.062477e-08,1.087778e-30,⋯,-0.8860290,-1.3264733,-0.4404444,-1.0589761,-0.8215709,0.2374052,-0.8486632,-1.4788452,-0.6301820,Acinar_1_2_6
7,MA0760.1_ERF,1.739912e-18,2.214106e-10,8.187445e-09,1.844295e-10,6.959646e-18,5.532885e-10,8.187445e-09,5.532885e-10,4.713012e-34,⋯,-0.9456788,-1.4157123,-0.4700335,-1.1281569,-0.8789790,0.2491779,-0.9087133,-1.5704964,-0.6617831,Acinar_1_2_6
8,MA0750.2_ZBTB7A,2.012863e-18,1.607868e-07,2.186183e-08,4.146400e-11,8.051453e-18,1.607868e-07,4.372365e-08,1.243920e-10,4.362193e-31,⋯,-0.9501258,-1.3880804,-0.4379546,-1.1383088,-0.8336639,0.3046449,-0.8880800,-1.6192694,-0.7311894,Acinar_1_2_6
9,MA1942.1_ETV2::FOXI1,4.382465e-18,4.751845e-14,9.723868e-11,7.725707e-09,1.752986e-17,1.425553e-13,1.944774e-10,7.725707e-09,2.047111e-34,⋯,-1.4900047,-2.3382243,-0.8482197,-1.8099317,-1.3951696,0.4147621,-1.4801468,-2.4355240,-0.9553772,Acinar_1_2_6


,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,MA0462.2_BATF::JUN,1.020715e-04,1.688774e-06,1.084321e-11,4.713687e-06,1.020715e-04,5.066322e-06,4.337286e-11,9.427373e-06,2.015332e-17,⋯,-0.8852479,-1.6956444,-0.8103965,-1.443273,-0.057321711,1.3859513,-0.7883377,-2.070221,-1.2818833,Acinar_3
2,MA1634.1_BATF,7.126106e-05,1.089571e-06,1.161588e-11,6.433320e-06,7.126106e-05,3.268713e-06,4.646354e-11,1.286664e-05,2.217783e-17,⋯,-0.8919592,-1.7259112,-0.8339520,-1.451226,-0.086590448,1.3646357,-0.8104452,-2.052857,-1.2424121,Acinar_3
3,MA0835.2_BATF3,6.112197e-05,6.253501e-07,1.338889e-11,3.340559e-06,6.112197e-05,1.876050e-06,5.355558e-11,6.681119e-06,5.300354e-18,⋯,-0.8561657,-1.6592256,-0.8030598,-1.395282,-0.078005962,1.3172764,-0.7769664,-1.976747,-1.1997801,Acinar_3
4,MA1928.1_BNC2,7.396547e-04,3.076904e-06,1.398722e-11,1.528436e-05,7.396547e-04,9.230712e-06,5.594888e-11,3.056873e-05,3.202807e-16,⋯,-0.8077191,-1.6132860,-0.8055669,-1.381229,0.068285772,1.4495145,-0.7100820,-1.990796,-1.2807142,Acinar_3
5,MA1138.1_FOSL2::JUNB,4.413773e-04,1.800921e-07,1.437781e-11,3.215151e-06,4.413773e-04,5.402763e-07,5.751122e-11,6.430302e-06,5.181466e-18,⋯,-0.7683412,-1.5567405,-0.7883993,-1.314440,0.057559626,1.3719993,-0.6830093,-1.891565,-1.2085553,Acinar_3
6,MA0501.1_MAF::NFE2,1.265579e-05,1.509500e-07,1.469702e-11,6.034259e-05,2.531158e-05,4.528500e-07,5.878809e-11,6.034259e-05,3.023022e-16,⋯,-0.4985581,-0.9759478,-0.4773897,-0.793457,-0.104086490,0.6893705,-0.4704330,-1.103308,-0.6328752,Acinar_3
7,MA1141.1_FOS::JUND,2.377327e-04,7.619808e-07,1.482023e-11,3.309391e-06,2.377327e-04,2.285942e-06,5.928090e-11,6.618783e-06,1.101902e-17,⋯,-0.9221540,-1.8115329,-0.8893789,-1.533985,-0.007945383,1.5260401,-0.8233141,-2.198349,-1.3750349,Acinar_3
8,MA0491.2_JUND,6.027011e-05,2.709517e-07,1.513020e-11,2.493698e-06,6.027011e-05,8.128551e-07,6.052079e-11,4.987397e-06,2.091325e-18,⋯,-0.8556839,-1.6920575,-0.8363736,-1.411024,-0.064897868,1.3461259,-0.7829964,-1.988949,-1.2059527,Acinar_3
9,MA0477.2_FOSL1,2.851182e-04,3.358983e-07,1.556007e-11,7.116509e-06,2.851182e-04,1.007695e-06,6.224029e-11,1.423302e-05,2.273613e-17,⋯,-0.8389594,-1.6725372,-0.8335778,-1.414215,0.018106763,1.4323218,-0.7500323,-2.021443,-1.2714106,Acinar_3


,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,MA0103.3_ZEB1,1.574728e-19,2.689590e-10,2.472934e-10,3.423745e-15,6.298912e-19,4.945867e-10,4.945867e-10,1.027123e-14,2.124091e-41,⋯,2.3721703,3.408943,1.0367728,2.8108584,2.1226393,-0.6882191,2.1942331,4.058333,1.8640995,Acinar_4
2,MA0745.2_SNAI2,8.832115e-20,8.553561e-10,3.539203e-15,7.046577e-15,3.532846e-19,8.553561e-10,1.061761e-14,1.409315e-14,3.368285e-42,⋯,2.4043512,3.480246,1.0758946,2.8764374,2.1061475,-0.7702898,2.2150132,4.169983,1.9549703,Acinar_4
3,MA1631.1_ASCL1,4.326072e-19,1.965950e-09,9.486600e-10,5.352134e-13,1.730429e-18,1.965950e-09,1.897320e-09,1.605640e-12,5.863048e-37,⋯,2.2543412,3.330059,1.0757174,2.7142743,1.9933736,-0.7209008,2.1144846,3.861668,1.7471839,Acinar_4
4,MA1559.1_SNAI3,8.164321e-20,7.540437e-09,6.905179e-12,5.997740e-13,3.265728e-19,7.540437e-09,1.381036e-11,1.799322e-12,1.195929e-37,⋯,2.4584340,3.579179,1.1207447,2.9600018,2.1244986,-0.8355032,2.2668351,4.279997,2.0131617,Acinar_4
5,MA0522.3_TCF3,7.807799e-20,4.058820e-09,1.232217e-09,8.678717e-13,3.123120e-19,4.058820e-09,2.464434e-09,2.603615e-12,5.413610e-37,⋯,2.1415102,3.120834,0.9793234,2.5592279,1.9062310,-0.6529969,1.9948947,3.666486,1.6715909,Acinar_4
6,MA0499.2_MYOD1,7.702942e-18,6.171311e-12,6.802537e-13,1.712024e-12,3.081177e-17,6.171311e-12,2.040761e-12,3.424049e-12,2.640159e-38,⋯,1.7903112,2.686590,0.8962785,2.1801882,1.5345805,-0.6456077,1.6744371,3.125558,1.4511214,Acinar_4
7,MA0830.2_TCF4,3.758440e-18,4.265788e-08,1.979291e-10,2.728476e-12,1.503376e-17,4.265788e-08,3.958583e-10,8.185429e-12,1.743239e-34,⋯,2.4904315,3.620064,1.1296328,2.9775802,2.2015483,-0.7760320,2.3146526,4.270708,1.9560549,Acinar_4
8,MA1529.1_NHLH2,1.749667e-15,2.003972e-07,6.815149e-14,4.192719e-12,6.998669e-15,2.003972e-07,2.044545e-13,8.385439e-12,4.587000e-33,⋯,0.6982516,1.015048,0.3167969,0.8702506,0.5141762,-0.3560744,0.6088347,1.324868,0.7160334,Acinar_4
9,MA1558.1_SNAI1,1.223894e-19,4.951820e-09,2.315334e-11,3.311749e-12,4.895578e-19,4.951820e-09,4.630669e-11,9.935248e-12,1.292575e-36,⋯,2.3981229,3.569641,1.1715185,2.9080490,2.1001217,-0.8079273,2.2490586,4.136897,1.8878384,Acinar_4


,Motif,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,anova.p.value,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,MA0757.1_ONECUT3,7.860170e-07,2.518007e-08,3.043517e-02,0.1929661008,2.358051e-06,1.007203e-07,6.087035e-02,0.192966101,7.019262e-07,⋯,0.4098005,0.7504370,0.3406365,0.5296189,0.3943250,-0.13529392,0.4561295,0.6275268,0.17139729,Acinar_5
2,MA1487.2_FOXE1,1.749221e-12,7.534956e-08,1.487714e-03,0.0008561141,6.996885e-12,2.260487e-07,1.712228e-03,0.001712228,5.497418e-18,⋯,0.5959919,1.0000988,0.4041069,0.7313921,0.6092861,-0.12210596,0.6318412,0.9190238,0.28718252,Acinar_5
3,MA0032.2_FOXC1,5.681457e-14,1.004736e-07,6.759203e-04,0.0082339261,2.272583e-13,3.014208e-07,1.351841e-03,0.008233926,1.709051e-16,⋯,0.8383168,1.4257605,0.5874437,1.0221162,0.8753280,-0.14678824,0.8990675,1.2754877,0.37642019,Acinar_5
4,MA0796.1_TGIF1,8.189617e-09,1.123886e-07,4.381274e-01,0.3539432911,3.275847e-08,3.371658e-07,7.078866e-01,0.707886582,1.174525e-08,⋯,0.3571668,0.6339666,0.2767998,0.4366181,0.4047033,-0.03191485,0.4096888,0.4895448,0.07985597,Acinar_5
5,MA0679.2_ONECUT1,1.604247e-08,2.311145e-07,9.086465e-03,0.0063969284,6.416990e-08,6.933435e-07,1.279386e-02,0.012793857,1.075043e-11,⋯,0.5249833,0.8380005,0.3130171,0.6335055,0.5057087,-0.12779683,0.5376136,0.8194710,0.28185735,Acinar_5
6,MA0046.2_HNF1A,1.364581e-08,2.705683e-07,1.618366e-03,0.0488345039,5.458326e-08,8.117049e-07,3.236732e-03,0.048834504,5.842930e-12,⋯,0.5003708,0.8859084,0.3855376,0.6372235,0.4949076,-0.14231585,0.5593840,0.7257210,0.16633704,Acinar_5
7,MA0153.2_HNF1B,1.189166e-08,3.622569e-07,2.705220e-04,0.0049901682,4.756665e-08,1.086771e-06,5.410440e-04,0.004990168,7.747732e-13,⋯,0.5624538,0.9215092,0.3590554,0.6979079,0.5345094,-0.16339845,0.5898299,0.8604052,0.27057521,Acinar_5
8,MA1574.1_THRB,9.948231e-12,3.845449e-07,1.800600e-01,0.0788297080,3.979292e-11,1.153635e-06,1.800600e-01,0.157659416,1.422039e-10,⋯,0.3350657,0.5600218,0.2249560,0.4033535,0.3626162,-0.04073727,0.3624936,0.4877944,0.12530076,Acinar_5
9,MA0845.1_FOXB1,3.893092e-14,8.419081e-07,1.509715e-05,0.0038604250,1.557237e-13,2.525724e-06,3.019429e-05,0.003860425,6.539704e-15,⋯,0.8451525,1.3943851,0.5492326,1.0368112,0.8286820,-0.20812924,0.8721682,1.3484419,0.47627367,Acinar_5


In [21]:
#Get families
TFClass_Lookup_sub <- select(TFClass_Lookup, Motif=full_jaspar_motif, Family=lowest_level_family)
TFClass_Lookup_sub <- TFClass_Lookup_sub[!duplicated(TFClass_Lookup_sub$Motif),]

TFClass_Lookup_sub

Motif,Family
<chr>,<chr>
MA0030.1_FOXF2,FOXF
MA0031.1_FOXD1,FOXD
MA0051.1_IRF2,IRF
MA0057.1_MZF1(var.2),More than 3 adjacent zinc fingers
MA0059.1_MAX::MYC,MYC
MA0066.1_PPARG,PPAR(NR1C)
MA0069.1_PAX6,PAX4-like
MA0070.1_PBX1,PBX
MA0071.1_RORA,ROR(NR1F)


In [22]:
# Add on families
dim(type_anova_results_full_holm_anova_dev)

type_anova_results_full_holm_anova_dev_fam <- select(left_join(type_anova_results_full_holm_anova_dev, TFClass_Lookup_sub), Motif, Family, everything())

dim(type_anova_results_full_holm_anova_dev_fam)
head(type_anova_results_full_holm_anova_dev_fam)

[1] 692  24

Joining with `by = join_by(Motif)`


[1] 692  25

,Motif,Family,Acinar_1_2_6.p.value,Acinar_5.p.value,Acinar_3.p.value,Acinar_4.p.value,Acinar_1_2_6.holm,Acinar_5.holm,Acinar_3.holm,Acinar_4.holm,⋯,not.Acinar_5,Acinar_5,Acinar_5.diff,not.Acinar_3,Acinar_3,Acinar_3.diff,not.Acinar_4,Acinar_4,Acinar_4.diff,max
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,MA0030.1_FOXF2,FOXF,1.878561e-06,7.161564e-03,0.4033856373,0.4695431201,7.514245e-06,2.148469e-02,0.8067712747,0.8067712747,⋯,0.33699151,0.51742727,0.18043576,0.37565291,0.40839287,0.03273996,0.36346805,0.44923345,0.08576540,Acinar_5
2,MA0031.1_FOXD1,FOXD,8.190823e-11,2.474089e-04,0.7115260632,0.0071438760,3.276329e-10,7.422268e-04,0.7115260632,0.0142877519,⋯,0.51777387,0.75361113,0.23583726,0.58349083,0.57171637,-0.01177446,0.51572609,0.78081496,0.26508887,Acinar_4
3,MA0051.1_IRF2,IRF,2.146152e-10,1.182946e-07,0.0001289029,0.0008358103,8.584606e-10,3.548837e-07,0.0002578058,0.0008358103,⋯,-0.45760182,-0.77134101,-0.31373919,-0.58512056,-0.40493006,0.18019050,-0.46521052,-0.77325198,-0.30804147,Acinar_1_2_6
4,MA0059.1_MAX::MYC,MYC,7.680233e-03,1.116966e-01,0.0013265279,0.1502414143,2.304070e-02,2.233933e-01,0.0053061117,0.2233932673,⋯,0.06217336,0.15690768,0.09473432,0.11512127,0.01296847,-0.10215279,0.06592540,0.15121380,0.08528841,Acinar_5
5,MA0066.1_PPARG,PPAR(NR1C),3.355028e-01,2.115934e-01,0.2765041439,0.6371728416,8.463736e-01,8.463736e-01,0.8463735940,0.8463735940,⋯,0.04754416,0.09334931,0.04580514,0.04802604,0.09448791,0.04646187,0.07143713,0.01943405,-0.05200309,Acinar_3
6,MA0069.1_PAX6,PAX4-like,1.016753e-06,6.478739e-03,0.0012452074,0.0280443027,4.067012e-06,1.295748e-02,0.0037356223,0.0280443027,⋯,-0.24760523,-0.39473373,-0.14712850,-0.31250140,-0.20176058,0.11074082,-0.23391655,-0.45161310,-0.21769655,Acinar_1_2_6


In [23]:
# Write it all out!
write.table(type_anova_results_full_holm_anova_dev_fam, 
            '/nfs/lab/welison/multiome/chromvar/240209_ChromVar_Outputs/Acinar_subtype_donor_anova_holm_results.tsv',
            row.names=F, col.names=T, quote=F, sep='\t')

write.table(filter(arrange(type_anova_results_full_holm_anova_dev_fam, Acinar_1_2_6.holm), 
              Acinar_1_2_6.holm < .05, anova.q.value < .05, max == 'Acinar_1_2_6'), 
            '/nfs/lab/welison/multiome/chromvar/240209_ChromVar_Outputs/Acinar_subtype_donor_anova_holm_results_Acinar_1_2_6_max.tsv',
            row.names=F, col.names=T, quote=F, sep='\t')
write.table(filter(arrange(type_anova_results_full_holm_anova_dev_fam, Acinar_3.holm), 
              Acinar_3.holm < .05, anova.q.value < .05, max == 'Acinar_3'), 
            '/nfs/lab/welison/multiome/chromvar/240209_ChromVar_Outputs/Acinar_subtype_donor_anova_holm_results_Acinar_3_max.tsv',
            row.names=F, col.names=T, quote=F, sep='\t')
write.table(filter(arrange(type_anova_results_full_holm_anova_dev_fam, Acinar_4.holm), 
              Acinar_4.holm < .05, anova.q.value < .05, max == 'Acinar_4'), 
            '/nfs/lab/welison/multiome/chromvar/240209_ChromVar_Outputs/Acinar_subtype_donor_anova_holm_results_Acinar_4_max.tsv',
            row.names=F, col.names=T, quote=F, sep='\t')
write.table(filter(arrange(type_anova_results_full_holm_anova_dev_fam, Acinar_5.holm), 
              Acinar_5.holm < .05, anova.q.value < .05, max == 'Acinar_5'), 
            '/nfs/lab/welison/multiome/chromvar/240209_ChromVar_Outputs/Acinar_subtype_donor_anova_holm_results_Acinar_5_max.tsv',
            row.names=F, col.names=T, quote=F, sep='\t')